# Missing Values and Duplicates Cleaning

In [50]:
import pandas as pd
import numpy as np
import re
import emoji 
import string

# Load dataset
df = pd.read_csv("MASTER_RAW_kenya_fintech.csv")

# Preview dataset
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,NaN,mpesa


## Checking Missing Values

This step identifies columns with null or missing values that may affect preprocessing and model training.

In [51]:
missing_values = df.isnull().sum()

missing_values

reviewId                    0
userName                    0
userImage                   0
content                     1
score                       0
thumbsUpCount               0
reviewCreatedVersion     6164
at                          0
replyContent            29574
repliedAt               29574
appVersion               6164
app_name                    0
dtype: int64

## Removing Empty Reviews

Rows with completely empty review texts are removed because they do not contribute meaningful information for sentiment analysis.

In [52]:
df = df[df['content'].notna()]

df.shape

(53506, 12)

## Handling Missing App Versions

Missing app versions are replaced with 'Unknown' to preserve dataset consistency.

In [53]:
df['appVersion'] = df['appVersion'].fillna('Unknown')

df['appVersion'].isnull().sum()

0

## Checking Duplicate Reviews

Duplicate reviews can bias the dataset and affect model learning. This step identifies repeated review texts.

In [54]:
duplicate_reviews = df.duplicated(subset=['content']).sum()

print("Duplicate Reviews:", duplicate_reviews)

Duplicate Reviews: 20278


In [55]:
df = df.drop_duplicates(subset=['content'])

df.shape

(33228, 12)

## Checking Duplicate Review IDs

Each review should have a unique review ID. Duplicate IDs may indicate repeated records.

In [56]:
duplicate_ids = df.duplicated(subset=['reviewId']).sum()

print("Duplicate Review IDs:", duplicate_ids)

Duplicate Review IDs: 0


In [57]:
df = df.drop_duplicates(subset=['reviewId'])

df.shape

(33228, 12)

## Checking Duplicate Users and Posts

This step checks whether the same users repeatedly posted identical reviews.

In [58]:
duplicate_users_posts = df.duplicated(
    subset=['userName', 'content']
).sum()

print("Duplicate Users/Posts:", duplicate_users_posts)

Duplicate Users/Posts: 0


## Removing Unnecessary Columns

Columns such as usernames and user images are removed because they are not required for sentiment classification.

In [59]:
df = df.drop(columns=['userImage', 'userName'])

df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa


## Final Dataset Validation

A final check is performed to confirm the dataset is clean and ready for preprocessing and modelling.

In [60]:
df.isnull().sum()

reviewId                    0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion     4356
at                          0
replyContent            19058
repliedAt               19058
appVersion                  0
app_name                    0
dtype: int64

In [61]:
df.to_csv("cleaned_kenya_fintech.csv", index=False)

print("Final cleaned dataset saved successfully.")

Final cleaned dataset saved successfully.


In [62]:
cleaned_df=pd.read_csv("cleaned_kenya_fintech.csv")
cleaned_df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa


## Implementing text normalization function

In [ ]:
def clean_text(text):
    text = str(text)  
    
    
    text = text.lower() # converting text to lowercase
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    
    
    text = re.sub(r'\d+', '', text)  #Removing numbers
    
    
    text = text.translate(str.maketrans('', '', string.punctuation))# Remove punctuations
    
    
    text = emoji.replace_emoji(text, replace='')# Remove emojis
    
    # Removing  special characters 
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()# remove spaces
    
    # Removing extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

## Applying the cleaning function

In [64]:
cleaned_df["cleaned_text"] = cleaned_df["content"].apply(clean_text)

cleaned_df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name,cleaned_text
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa,the app still has issues on otp because i have...
1,acd5c061-de13-474b-8645-f628044f2a50,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa,si everytime nitakuwa na bundles za ku check m...
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa,this is the stupidest app ever from saf the worst
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa,life must go on without this useless app it us...
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,Unknown,mpesa,the upgrade is terrible


The text data was preprocessed by converting it to lowercase, removing URLs, emojis, and non-alphabetic characters, and eliminating extra spaces to ensure consistency. The cleaned data is now suitable for further analysis and machine learning.